< DataFrame 다루기 - 결측치(누락값) 처리 2 >

In [1]:
# 1. 시계열 데이터의 결측치 
# 시계열 데이터의 경우 시간의 흐름이 중요하므로, 같은 시기의 데이터를 활용하거나 해당 기간의 평균/최빈값으로 대체

import pandas as pd

data = {
    "날짜": [
        "2025-01-01", "2025-01-02", "2025-01-03",
        "2025-02-01", "2025-02-02", "2025-02-03",
        "2025-03-01", "2025-03-02", "2025-03-03"
    ],
    "매출": [
        100, 120, None,
        200, None, 220,
        300, 320, None
    ]
}

df = pd.DataFrame(data)

df["날짜"] = pd.to_datetime(df["날짜"])

print(df)

          날짜     매출
0 2025-01-01  100.0
1 2025-01-02  120.0
2 2025-01-03    NaN
3 2025-02-01  200.0
4 2025-02-02    NaN
5 2025-02-03  220.0
6 2025-03-01  300.0
7 2025-03-02  320.0
8 2025-03-03    NaN


In [2]:
# *.dt : 날짜 데이터에서 연·월·일·요일·시간 등을 뽑거나 날짜 관련 처리를 하기 위한 pandas 전용 accessor

# ① 월 파생변수 생성 → ② 월별 평균 계산 → ③ 해당 월의 평균으로 결측치 대체
# 이 후에 groupby() 메서드를 배우면 더 간단히 할 수 있다.

df["월"] = df["날짜"].dt.month
print(df)

          날짜     매출  월
0 2025-01-01  100.0  1
1 2025-01-02  120.0  1
2 2025-01-03    NaN  1
3 2025-02-01  200.0  2
4 2025-02-02    NaN  2
5 2025-02-03  220.0  2
6 2025-03-01  300.0  3
7 2025-03-02  320.0  3
8 2025-03-03    NaN  3


In [3]:
jan_mean = df.loc[df["월"] == 1, "매출"].mean()
feb_mean = df.loc[df["월"] == 2, "매출"].mean()
mar_mean = df.loc[df["월"] == 3, "매출"].mean()

print("1월 평균:", jan_mean)
print("2월 평균:", feb_mean)
print("3월 평균:", mar_mean)

1월 평균: 110.0
2월 평균: 210.0
3월 평균: 310.0


In [4]:
df.loc[(df["월"] == 1) & (df["매출"].isna()), "매출"] = jan_mean
df.loc[(df["월"] == 2) & (df["매출"].isna()), "매출"] = feb_mean
df.loc[(df["월"] == 3) & (df["매출"].isna()), "매출"] = mar_mean
df

,날짜,매출,월
0,2025-01-01,100.0,1
1,2025-01-02,120.0,1
2,2025-01-03,110.0,1
3,2025-02-01,200.0,2
4,2025-02-02,210.0,2
5,2025-02-03,220.0,2
6,2025-03-01,300.0,3
7,2025-03-02,320.0,3
8,2025-03-03,310.0,3


2. 회귀 알고리즘을 활용해 다른 변수와의 관계를 바탕으로 결측값을 예측하여 대체

- scikit-learn 라이브러리 설치
Python에서 머신러닝을 쉽게 구현할 수 있도록 만든 라이브러리
(.venv) pip install scikit-learn

In [5]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [6]:
import pandas as pd

data = {
    "공부시간": [1, 2, 3, 4, 5, 6, 7, 8],
    "게임시간": [5, 4, 4, 3, 3, 2, 2, 1],
    "시험점수": [45, 50, None, 62, 68, None, 80, 88]
}

df = pd.DataFrame(data)

df

,공부시간,게임시간,시험점수
0,1,5,45.0
1,2,4,50.0
2,3,4,NaN
3,4,3,62.0
4,5,3,68.0
5,6,2,NaN
6,7,2,80.0
7,8,1,88.0


In [7]:
train = df[df["시험점수"].notna()] # 시험점수가 있는 데이터만 학습에 사용

X_train = train[["공부시간", "게임시간"]]
y_train = train["시험점수"]

print(X_train)
print(y_train)

   공부시간  게임시간
0     1     5
1     2     4
3     4     3
4     5     3
6     7     2
7     8     1
0    45.0
1    50.0
3    62.0
4    68.0
6    80.0
7    88.0
Name: 시험점수, dtype: float64
